## Table 1 / presence of species in things

All files on farm in `/home/ctbrown/scratch3/2025-workflow-core99/`

TODO: add E. coli

In [1]:
# base checkout of workflow directory is here:
BASE='/home/ctbrown/scratch3/2025-workflow-core99/'

# parquet files from 'sourmash gather' against 3,216 metagenomes
BASE_OUTPUTS=BASE+'/outputs.core2'

# where to place figures
FIG_OUT=BASE+'/outputs.figures/'

THRESHOLD_BP=20_000
N_METAGENOMES=3216

In [2]:
import polars as pl
import glob
import matplotlib.pyplot as plt

In [3]:
extract_species_from_name = (pl.col("match_name")
                             .str.split(' ')
                             .list.slice(1, 2)
                             .list.join(' '))

In [4]:
names_core = set([ x.strip() for x in open('../inputs.cds/names.list') ])
names_large = set([ x.strip() for x in open('../inputs.cds/names-plus.list') ])

names_core.add('s__Escherichia coli')
names_large.add('s__Escherichia coli')

In [5]:
filenames = glob.glob(BASE_OUTPUTS + '/*.parquet')

dflist = []

all_df = (pl.scan_parquet(filenames)
          .with_columns(species=extract_species_from_name)
          .filter(pl.col("species").is_in(names_large))
          .select(["species", "match_name", "containment", "query_name", "intersect_hashes", "scaled"])).collect()


In [6]:
assert all_df["query_name"].n_unique() == N_METAGENOMES
print(f"loaded {all_df['query_name'].n_unique()} gathers.")

loaded 3216 gathers.


In [7]:
all_df = all_df.with_columns((pl.col("intersect_hashes") * pl.col("scaled")).alias("intersect_bp"))

In [8]:
all_df = all_df.filter(pl.col("intersect_bp") >= THRESHOLD_BP)

In [9]:
group_df = (all_df.group_by('species')
            .agg(pl.len())
            .with_columns(frequency=pl.col("len") / N_METAGENOMES))

            
group_df

species,len,frequency
str,u32,f64
"""s__Bifidobacterium thermacidop…",1582,0.491915
"""s__Cryptobacteroides sp0340892…",3111,0.967351
"""s__UMGS1225 sp034086705""",2533,0.787624
"""s__CAG-269 sp014846485""",1014,0.315299
"""s__Phascolarctobacterium_A suc…",891,0.277052
…,…,…
"""s__CAG-873 sp016302215""",1130,0.351368
"""s__Blautia_A sp945872375""",1168,0.363184
"""s__Sodaliphilus sp004557565""",3188,0.991294


In [10]:
cds3_df = (pl.scan_csv('../outputs.cds/cds3/manysearch.cds3.3216.csv')
           .with_columns(species=pl.col('query_name'), intersect_bp=pl.col("scaled") * pl.col("intersect_hashes"))
           .filter(pl.col("intersect_bp") >= THRESHOLD_BP)
           .select(["species", "match_name", "containment", "intersect_hashes", "scaled"])).collect()

cds3_df = (cds3_df.group_by('species')
            .agg(pl.len())
            .with_columns(freq_cds3=pl.col("len") / N_METAGENOMES)
            .select(['species', 'freq_cds3']))

            
cds3_df

species,freq_cds3
str,f64
"""s__Cryptobacteroides sp0340892…",0.96393
"""s__Sodaliphilus sp004557565""",0.989739
"""s__Ornithospirochaeta sp022785…",0.952114
"""s__UMGS1225 sp034086705""",0.776119
"""s__Lactobacillus amylovorus""",0.984764
…,…
"""s__Fimisoma sp002320005""",0.970771
"""s__Mogibacterium_A kristiansen…",0.970149
"""s__UBA2868 sp004552595""",0.96704


In [11]:
join_df = group_df.join(cds3_df, on='species', how='inner')

export_df = (join_df.with_columns(
    species_m=pl.when(
        pl.col("species").is_in(names_core)
    ).then(pl.concat_str(pl.lit('*'), pl.col("species")))
    .otherwise(pl.col("species")))
    .sort(by='frequency', descending=True)
    .select(['species_m', 'frequency', 'freq_cds3'])
)

In [12]:
with pl.Config(tbl_rows=-1):
    print(export_df)

shape: (27, 3)
┌─────────────────────────────────┬───────────┬───────────┐
│ species_m                       ┆ frequency ┆ freq_cds3 │
│ ---                             ┆ ---       ┆ ---       │
│ str                             ┆ f64       ┆ f64       │
╞═════════════════════════════════╪═══════════╪═══════════╡
│ *s__Sodaliphilus sp004557565    ┆ 0.991294  ┆ 0.989739  │
│ *s__Lactobacillus amylovorus    ┆ 0.989739  ┆ 0.984764  │
│ *s__Mogibacterium_A kristianse… ┆ 0.981032  ┆ 0.970149  │
│ *s__UBA2868 sp004552595         ┆ 0.9801    ┆ 0.96704   │
│ *s__Cryptobacteroides sp900546… ┆ 0.978545  ┆ 0.975124  │
│ *s__JAFBIX01 sp021531895        ┆ 0.975124  ┆ 0.962998  │
│ *s__Fimisoma sp002320005        ┆ 0.973881  ┆ 0.970771  │
│ *s__Floccifex porci             ┆ 0.971393  ┆ 0.964552  │
│ *s__Prevotella sp002251295      ┆ 0.968595  ┆ 0.963619  │
│ *s__Cryptobacteroides sp034089… ┆ 0.967351  ┆ 0.96393   │
│ *s__Bariatricus sp004560705     ┆ 0.964241  ┆ 0.9574    │
│ *s__Colivicinus sp00229

In [13]:
export_df.write_csv('../outputs.cds/core-summary.csv')